## Goal

In [1]:
# importing libraries
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import xarray as xr
import lightgbm as lgb
print(lgb.__file__)
print(lgb.__version__)

c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\lightgbm\__init__.py
4.6.0


In [2]:
# setting directory so code will function on any machine
repo_root = Path.cwd().resolve().parent
dir = repo_root / "data" / "processed"
model_out = repo_root / "models" / "lgbm_t2m_sst.txt"

# defining key variables to be used to train the model
time_gap = 3
keep_row = 0.10
land_threshold = 0.5
n_test_years = 2
seed = 42

In [3]:
# checking to see if all files are there
files = sorted(glob.glob(str(dir / "*.nc")))
if not files:
    raise FileNotFoundError(f"No .nc files under {dir}")
print(f"Found {len(files)} files")

# extracting latitude and longitude variables
# taking the first values of static variables (land-sea mask and geopotential)
with xr.open_dataset(files[0], engine="netcdf4") as first:
    lat = first["latitude"].values.astype(np.float32)
    lon = first["longitude"].values.astype(np.float32)
    lsm2d = first["lsm"].isel(valid_time=0).values.astype(np.float32)
    z2d = first["z"].isel(valid_time=0).values.astype(np.float32)

# calculating number of grid cells and creating flattened arrays for lat, lon, lsm and z
n_cell = lat.size * lon.size
lon2d, lat2d = np.meshgrid(lon, lat)
lat_flat = lat2d.ravel()
lon_flat = lon2d.ravel()
lsm_flat = lsm2d.ravel()
z_flat = z2d.ravel()
is_land = lsm_flat >= land_threshold
print(f"Grid: {lat.size} x {lon.size} = {n_cell:,} cells "
      f"({int(is_land.sum()):,} land)")

# introducing variables to be used in the model
vars = ["ssrd", "strd", "tcc", "lsm", "z",
            "lat", "lon", "hour_sin", "hour_cos", "doy_sin", "doy_cos"]

Found 120 files
Grid: 125 x 121 = 15,125 cells (8,385 land)


In [4]:
rng = np.random.default_rng(seed)
x_parts, y_parts, yr_parts = [], [], []

for i, f in enumerate(files, 1):
    with xr.open_dataset(f, engine="netcdf4") as blk:
        blk = blk.isel(valid_time=slice(None, None, time_gap))  # keep every 3rd hour
        nb = blk.sizes["valid_time"]

        t2m = blk["t2m"].values.astype(np.float32).reshape(nb, -1)
        sst = blk["sst"].values.astype(np.float32).reshape(nb, -1)
        ssrd = blk["ssrd"].values.astype(np.float32).reshape(nb, -1)
        strd = blk["strd"].values.astype(np.float32).reshape(nb, -1)
        tcc = blk["tcc"].values.astype(np.float32).reshape(nb, -1)
        times = pd.DatetimeIndex(blk["valid_time"].values)

    # Target to align t2m over land, sst over sea
    target = np.where(is_land[None, :], t2m, sst)

    hour = times.hour.to_numpy(dtype=np.float32)
    doy = times.dayofyear.to_numpy(dtype=np.float32)
    year = times.year.to_numpy(dtype=np.int16)

    # Cyclic time encoding
    h_sin = np.sin(2 * np.pi * hour / 24).astype(np.float32)
    h_cos = np.cos(2 * np.pi * hour / 24).astype(np.float32)
    d_sin = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    d_cos = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

    def tile_t(a):          # per-timestep value is (nb, n_cell)
        return np.repeat(a[:, None], n_cell, axis=1)

    def tile_c(a):          # per-cell value is (nb, n_cell)
        return np.repeat(a[None, :], nb, axis=0)

    xb = np.stack([ssrd, strd, tcc,
                   tile_c(lsm_flat), tile_c(z_flat),
                   tile_c(lat_flat), tile_c(lon_flat),
                   tile_t(h_sin), tile_t(h_cos),
                   tile_t(d_sin), tile_t(d_cos)], axis=-1)

    xb = xb.reshape(-1, len(vars))
    yb = target.ravel()
    yrb = np.repeat(year, n_cell)

    keep = ~np.isnan(yb)
    keep &= rng.random(yb.size) < keep_row
    x_parts.append(xb[keep])
    y_parts.append(yb[keep])
    yr_parts.append(yrb[keep])
    if i % 12 == 0 or i == len(files):
        print(f"  {i}/{len(files)} files, "
              f"{sum(p.shape[0] for p in x_parts):,} rows so far")

x = np.concatenate(x_parts)
y = np.concatenate(y_parts)
yrs = np.concatenate(yr_parts)
del x_parts, y_parts, yr_parts
print(f"Total training table: {x.shape[0]:,} rows x {x.shape[1]} features "
      f"({x.nbytes / 1e9:.2f} GB)")

  12/120 files, 4,379,829 rows so far
  24/120 files, 8,750,538 rows so far
  36/120 files, 13,104,936 rows so far
  48/120 files, 17,557,098 rows so far
  60/120 files, 21,998,098 rows so far
  72/120 files, 26,415,788 rows so far
  84/120 files, 30,845,738 rows so far
  96/120 files, 35,276,172 rows so far
  108/120 files, 39,693,689 rows so far
  120/120 files, 44,081,694 rows so far
Total training table: 44,081,694 rows x 11 features (1.94 GB)


In [5]:
uyears = np.unique(yrs)
test_years = uyears[-n_test_years:]
val_year = uyears[-n_test_years - 1]
train_years = uyears[: -n_test_years - 1]
print(f"train={list(train_years)} val={val_year} test={list(test_years)}")

tr = np.isin(yrs, train_years)
va = yrs == val_year
te = np.isin(yrs, test_years)

train=[np.int16(2016), np.int16(2017), np.int16(2018), np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022)] val=2023 test=[np.int16(2024), np.int16(2025)]


In [6]:
model = lgb.LGBMRegressor(
    objective="regression", n_estimators=3000, learning_rate=0.05,
    num_leaves=127, feature_fraction=0.9, subsample=0.8, subsample_freq=1,
    n_jobs=-1, random_state=seed, verbose=-1,
)
model.fit(
    x[tr], y[tr],
    eval_set=[(x[va], y[va])], eval_metric="rmse",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)

Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 1.77008	valid_0's l2: 3.13319
[400]	valid_0's rmse: 1.72205	valid_0's l2: 2.96545
[600]	valid_0's rmse: 1.70319	valid_0's l2: 2.90087
[800]	valid_0's rmse: 1.69297	valid_0's l2: 2.86615
[1000]	valid_0's rmse: 1.68399	valid_0's l2: 2.83583
[1200]	valid_0's rmse: 1.67749	valid_0's l2: 2.81398
[1400]	valid_0's rmse: 1.6722	valid_0's l2: 2.79627
[1600]	valid_0's rmse: 1.66794	valid_0's l2: 2.78201
[1800]	valid_0's rmse: 1.66534	valid_0's l2: 2.77334
[2000]	valid_0's rmse: 1.66236	valid_0's l2: 2.76345
[2200]	valid_0's rmse: 1.65973	valid_0's l2: 2.75471
[2400]	valid_0's rmse: 1.65779	valid_0's l2: 2.74826
[2600]	valid_0's rmse: 1.65606	valid_0's l2: 2.74255
[2800]	valid_0's rmse: 1.65496	valid_0's l2: 2.73888
[3000]	valid_0's rmse: 1.65338	valid_0's l2: 2.73366
Did not meet early stopping. Best iteration is:
[3000]	valid_0's rmse: 1.65338	valid_0's l2: 2.73366


,num_leaves,127
,learning_rate,0.05
,n_estimators,3000
,objective,'regression'
,subsample,0.8
,subsample_freq,1
,random_state,42
,n_jobs,-1
,feature_fraction,0.9
,verbose,-1
,boosting_type,'gbdt'


In [7]:
def report(name, y_true, y_pred):
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err ** 2))
    r2 = 1 - np.sum(err ** 2) / np.sum((y_true - y_true.mean()) ** 2)
    print(f"{name:<22} MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.4f}")

y_hat = model.predict(x[te], num_iteration=model.best_iteration_)
report("LightGBM", y[te], y_hat)

lat_i, lon_i, hs_i, hc_i, ds_i, dc_i = 5, 6, 7, 8, 9, 10
def clim_key(xm):
    hour = np.round(np.degrees(np.arctan2(xm[:, hs_i], xm[:, hc_i]))
                    / 15.0) % 24
    doy = (np.degrees(np.arctan2(xm[:, ds_i], xm[:, dc_i])) % 360) / 360 * 365.25
    month = np.clip((doy // 30.5).astype(int), 0, 11)
    return pd.MultiIndex.from_arrays(
        [xm[:, lat_i].round(2), xm[:, lon_i].round(2), month, hour.astype(int)])

clim = pd.Series(y[tr], index=clim_key(x[tr])).groupby(level=[0, 1, 2, 3]).mean()
base = clim.reindex(clim_key(x[te])).to_numpy()
ok = ~np.isnan(base)
report("Climatology baseline", y[te][ok], base[ok])
report("LightGBM (same rows)", y[te][ok], y_hat[ok])

imp = pd.DataFrame({"feature": vars,
                    "gain": model.booster_.feature_importance("gain")}
                   ).sort_values("gain", ascending=False)
print(imp.to_string(index=False))

model_out.parent.mkdir(parents=True, exist_ok=True)
model.booster_.save_model(str(model_out))
print(f"Saved model -> {model_out}")

c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM               MAE=1.009  RMSE=1.569  R2=0.9847
Climatology baseline   MAE=1.436  RMSE=2.246  R2=0.9687
LightGBM (same rows)   MAE=1.009  RMSE=1.569  R2=0.9847
 feature         gain
    strd 3.303012e+10
       z 4.283556e+09
     tcc 8.496915e+08
 doy_cos 8.133110e+08
    ssrd 6.097201e+08
hour_cos 4.794075e+08
     lat 4.364525e+08
     lsm 2.963207e+08
     lon 2.951330e+08
 doy_sin 2.621571e+08
hour_sin 9.126400e+07
Saved model -> C:\Users\idasi\Documents\ideas_tih\IDEAS-TIH-Summer-2026\models\lgbm_t2m_sst.txt
